# 05 · Copula Tail Dependence Analysis
**Brazilian Stock-Bond Correlation Study**

Tests whether Brazilian asset pairs exhibit **asymmetric co-crash behaviour**:
do bonds and stocks crash *together* more than they boom together?

1. Transform returns to uniform pseudo-observations
2. Fit four copulas: Gaussian, Student-t, Clayton, Gumbel
3. Compare fit via AIC/BIC
4. Compute lower-tail dependence coefficient λ_L
5. Visualise joint tail behaviour

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import rankdata, t as t_dist

from fetch import load_master, CRISES

master = load_master()
plt.rcParams.update({
    "figure.dpi":150,"figure.facecolor":"white",
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.3,"font.size":11,
})
CRISIS_COLORS = {
    "GFC":"#d62728","Dilma":"#ff7f0e","Joesley":"#9467bd",
    "COVID":"#2ca02c","Americanas":"#8c564b","Fiscal24":"#e377c2",
}

# All copula densities live in src/metrics.py and are unit-tested (tests/test_metrics.py):
#   - each density integrates to 1 over the unit square
#   - Gumbel collapses to c == 1 at theta = 1 (the independence copula)
#   - Student-t converges to Gaussian as nu -> infinity
#   - fitting Clayton-simulated data recovers Clayton and its theta
# Getting a copula density wrong does not raise an error, it silently returns a
# likelihood for a function that is not a density -- and then AIC picks that family.
from metrics import (pseudo_obs, fit_all_copulas, tail_dependence_empirical,
                     gaussian_logpdf, student_t_logpdf, clayton_logpdf, gumbel_logpdf,
                     fit_gaussian, fit_student_t, fit_clayton, fit_gumbel)

# sanity check, cheap and worth running every time
import numpy as _np
_g = (_np.arange(200) + 0.5) / 200
_U, _V = _np.meshgrid(_g, _g)
_mass = _np.exp(gumbel_logpdf(2.0, _U.ravel(), _V.ravel())).sum() / 200**2
assert abs(_mass - 1) < 0.05, f"Gumbel density does not integrate to 1 ({_mass:.3f})"
assert _np.allclose(gumbel_logpdf(1.0, _U.ravel(), _V.ravel()), 0.0, atol=1e-9)
print("Copula densities OK (integrate to 1; Gumbel -> independence at theta=1)")

## 2. Fit and compare all copulas — Ibovespa × NTN-B

In [ ]:
df_pair = master[["ibov","ntnb"]].dropna() * 100
u_df    = pseudo_obs(df_pair)
u, v    = u_df["ibov"].values, u_df["ntnb"].values
n       = len(u)

print(f"Fitting copulas (Ibovespa x NTN-B 5y, n={n:,})...")
fits = fit_all_copulas(u, v)          # returned already sorted by AIC

rows = [{
    "Copula":    f["family"],
    "Log-lik":   round(f["ll"], 1),
    "AIC":       round(f["AIC"], 1),
    "BIC":       round(f["BIC"], 1),
    "lambda_L":  round(f["lambda_L"], 4),
    "lambda_U":  round(f["lambda_U"], 4),
    "Key param": f["param"],
} for f in fits]

fit_tbl = pd.DataFrame(rows).set_index("Copula")
print(fit_tbl.to_string())
fit_tbl.to_csv("../outputs/nb_tbl_copula_fit.csv")
print("\nBest fit (lowest AIC):", fit_tbl['AIC'].idxmin())
print("\nParametric tail dependence:")
for n_, row in fit_tbl.iterrows():
    print(f"  {n_:<12} lambda_L = {row['lambda_L']:.4f}   lambda_U = {row['lambda_U']:.4f}")

# The parametric lambdas are only as good as the family that wins AIC. The empirical
# exceedance rate makes no distributional assumption at all, so quote both.
emp = tail_dependence_empirical(u, v, q=0.05)
print(f"\nEmpirical 5% tail dependence (independence benchmark = 0.050):")
print(f"  lambda_L = {emp['lambda_L']:.3f}   lambda_U = {emp['lambda_U']:.3f}")
print(f"  co-crash observations: {emp['n_co_lower']} vs {emp['expected_indep']:.0f} "
      f"expected under independence ({emp['n_co_lower']/emp['expected_indep']:.1f}x)")

## 3. Tail dependence across all asset pairs

In [ ]:
bond_pairs = [("ibov","ntnb"),("ibov","ltn"),("ibov","ntnf"),("ibov","lft")]
LABELS = {"ibov":"Ibovespa","ntnb":"NTN-B 5y","ltn":"LTN 2y",
          "ntnf":"NTN-F 10y","lft":"LFT 1y"}

summary_rows = []
for ca, cb in bond_pairs:
    df_p = master[[ca,cb]].dropna() * 100
    uu   = pseudo_obs(df_p)
    ui, vi = uu[ca].values, uu[cb].values

    fits_p = fit_all_copulas(ui, vi)
    best   = fits_p[0]
    emp_p  = tail_dependence_empirical(ui, vi, q=0.05)

    summary_rows.append({
        "Pair":            f"{LABELS[ca]} x {LABELS[cb]}",
        "n":               len(ui),
        "Best copula":     best["family"],
        "Best param":      best["param"],
        "lambda_L (fit)":  round(best["lambda_L"], 4),
        "lambda_U (fit)":  round(best["lambda_U"], 4),
        "lambda_L (emp)":  round(emp_p["lambda_L"], 3),
        "lambda_U (emp)":  round(emp_p["lambda_U"], 3),
        "co-crash obs":    emp_p["n_co_lower"],
        "expected indep":  round(emp_p["expected_indep"], 0),
    })
    print(f"{LABELS[ca]} x {LABELS[cb]:<10} best={best['family']:<10} {best['param']:<22} "
          f"emp lambda_L={emp_p['lambda_L']:.3f} lambda_U={emp_p['lambda_U']:.3f}  "
          f"co-crash {emp_p['n_co_lower']} vs {emp_p['expected_indep']:.0f}")

print("\nlambda_L > lambda_U means the pair crashes together more than it booms together.")
print("Both are compared against 0.050, the rate implied by independence.")

summary_df = pd.DataFrame(summary_rows).set_index("Pair")
summary_df.to_csv("../outputs/nb_tbl_tail_dependence.csv")
print("\nSaved: outputs/nb_tbl_tail_dependence.csv")
print(summary_df.to_string())

## 4. Pseudo-observations scatter with tail quadrant analysis — Figure 7

In [ ]:
bond_cols = ["ntnb","ltn","ntnf","lft"]
colors    = ["#d62728","#ff7f0e","#2ca02c","#9467bd"]

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
axes = axes.flatten()

for i, (col, color) in enumerate(zip(bond_cols, colors)):
    ax = axes[i]
    df_p = master[["ibov", col]].dropna() * 100
    uu   = pseudo_obs(df_p)
    ui, vi = uu["ibov"].values, uu[col].values
    crisis_labels = master["crisis"].reindex(df_p.index).fillna("None")

    # Base scatter (grey)
    ax.scatter(ui, vi, s=3, color="#cccccc", alpha=0.3, zorder=1)

    # Highlight crisis observations
    for cname in CRISES:
        mask = crisis_labels == cname
        if mask.sum() > 0:
            ax.scatter(ui[mask.values], vi[mask.values],
                       s=15, color=CRISIS_COLORS[cname],
                       alpha=0.8, zorder=2, label=cname)

    # Mark tail quadrants (bottom-left = co-crash)
    q_lo = 0.10
    ax.axvline(q_lo, color="black", ls="--", lw=0.7, alpha=0.5)
    ax.axhline(q_lo, color="black", ls="--", lw=0.7, alpha=0.5)

    # Count observations in lower-left quadrant
    in_ll = ((ui < q_lo) & (vi < q_lo)).sum()
    expected_indep = len(ui) * q_lo**2
    ax.text(0.02, 0.12,
            f"Co-crash\nobserved: {in_ll}\nexpected (indep): {expected_indep:.0f}",
            transform=ax.transAxes, fontsize=8.5, va="bottom",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#d62728", alpha=0.8))

    # Annotate with the empirical exceedance rates (no distributional assumption)
    emp_i = tail_dependence_empirical(ui, vi, q=0.10)
    best_i = fit_all_copulas(ui, vi)[0]
    ax.set_title(f"Ibovespa x {LABELS[col]}\n"
                 f"best fit: {best_i['family']} ({best_i['param']})\n"
                 f"empirical 10% tail: lambda_L={emp_i['lambda_L']:.3f}  "
                 f"lambda_U={emp_i['lambda_U']:.3f}  (indep = 0.100)",
                 fontsize=9)
    ax.set_xlabel("Ibovespa (pseudo-obs u)", fontsize=9)
    ax.set_ylabel(f"{LABELS[col]} (pseudo-obs v)", fontsize=9)
    if i == 0:
        ax.legend(fontsize=7.5, loc="upper left")

fig.suptitle("Copula pseudo-observations: joint tail behaviour\n"
             "(lower-left quadrant = simultaneous crashes, "
             "dashed lines = 10th percentile thresholds)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/fig_copula_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_copula_scatter.png")

## ✅ Notebook 05 complete

**How to read the output above.** Two tail-dependence estimates are reported and they
answer slightly different questions:

- **Parametric λ** comes from whichever family wins on AIC. It is only meaningful if
  that family actually describes the data, so it inherits all of the family's
  assumptions — a Student-t fit, for instance, *imposes* λ_L = λ_U and therefore
  cannot detect asymmetry even if it is present.
- **Empirical λ** is the raw exceedance rate P(V < q | U < q), with no distributional
  assumption. Compare it against `q` itself, which is the rate implied by independence.
  This is the estimate to quote when the question is "do these assets crash together
  more often than chance?"

Where the two disagree, prefer the empirical one and say so. Reporting only the
parametric λ from a mis-specified family is how a copula analysis ends up asserting
the opposite of what the data show.

**Next:** `06_portfolio_metrics.ipynb` — Diversification Ratio, ENB, PCA